# Phase 2: Feature Engineering (RFM Analysis)
**Author:** Anjana Herath
This notebook loads the cleaned transaction data, calculates the Recency, Frequency, and Monetary (RFM) values for each customer, and applies Log Transformation and Standard Scaling to prepare the data for K-Means Clustering.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# Create outputs directory if it doesn't exist
os.makedirs('../outputs', exist_ok=True)

In [ ]:
# 1. Load Nayanajith's cleaned dataset
df = pd.read_csv('../outputs/1_cleaned_data_nayanajith.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"Loaded {len(df):,} clean transactions.")

In [ ]:
# 2. Calculate Recency, Frequency, and Monetary (RFM)
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days, # Recency
    'Invoice': 'nunique',                                     # Frequency
    'TotalCost': 'sum'                                        # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
rfm.head()

In [ ]:
# 3. Apply Log Transformation & Standard Scaling
rfm['Log_R'] = np.log1p(rfm['Recency'])
rfm['Log_F'] = np.log1p(rfm['Frequency'])
rfm['Log_M'] = np.log1p(rfm['Monetary'])

scaler = StandardScaler()
rfm[['Scaled_R', 'Scaled_F', 'Scaled_M']] = scaler.fit_transform(rfm[['Log_R', 'Log_F', 'Log_M']])

rfm.head()

In [ ]:
# 4. Save the engineered data for Liyanage & Sub-Team B
rfm.to_csv('../outputs/2_rfm_data_herath.csv', index=False)
print("Engineered data successfully saved!")